In [ ]:
# Install dependencies
!pip install -q gradio chromadb pypdf sentence-transformers google-generativeai

import gradio as gr
import chromadb
import google.generativeai as genai
import uuid
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# ==============================
# Configure Gemini API
# ==============================
genai.configure(api_key="YOUR_GEMINI_API_KEY")   # <-- Replace with your API key

llm = genai.GenerativeModel("gemini-2.5-flash")

# ==============================
# Load Embedding Model
# ==============================
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded!")

# ==============================
# ChromaDB
# ==============================
client = chromadb.Client()
collection = client.get_or_create_collection("rag_docs")


# ==============================
# Extract text from PDF
# ==============================
def extract_text(pdf):
    reader = PdfReader(pdf.name)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    return text


# ==============================
# Chunk text
# ==============================
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks


# ==============================
# Upload PDF
# ==============================
def upload_pdf(pdf):
    global collection

    if pdf is None:
        return "Please upload a PDF."

    text = extract_text(pdf)

    if len(text.strip()) == 0:
        return "No readable text found in the PDF."

    chunks = chunk_text(text)

    embeddings = embedder.encode(chunks).tolist()

    # Delete previous collection
    try:
        client.delete_collection("rag_docs")
    except:
        pass

    collection = client.get_or_create_collection("rag_docs")

    collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=[str(uuid.uuid4()) for _ in chunks]
    )

    return f"✅ PDF Indexed Successfully!\nChunks Created: {len(chunks)}"


# ==============================
# Ask Question
# ==============================
def ask(question):

    if collection.count() == 0:
        return "Please upload and index a PDF first."

    query_embedding = embedder.encode(question).tolist()

    result = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    docs = result.get("documents", [])

    if not docs or not docs[0]:
        return "I couldn't find the answer in the document."

    context = "\n".join(docs[0])

    context = context[:4000]

    prompt = f"""
You are a helpful AI assistant.

Answer ONLY using the context below.

If the answer is not present in the context, reply exactly:

I couldn't find the answer in the document.

Context:
{context}

Question:
{question}

Answer:
"""

    try:
        response = llm.generate_content(prompt)
        return response.text

    except Exception as e:
        return f"Error: {str(e)}"


# ==============================
# Gradio UI
# ==============================
with gr.Blocks(title="RAG PDF Chatbot") as demo:

    gr.Markdown("# 📄 RAG PDF Chatbot")
    gr.Markdown("Upload a PDF and ask questions from it.")

    pdf = gr.File(label="Upload PDF")

    upload_btn = gr.Button("Index PDF")

    status = gr.Textbox(label="Status")

    upload_btn.click(
        upload_pdf,
        inputs=pdf,
        outputs=status
    )

    question = gr.Textbox(
        label="Ask a Question",
        placeholder="Type your question here..."
    )

    ask_btn = gr.Button("Ask")

    answer = gr.Textbox(
        label="Answer",
        lines=10
    )

    ask_btn.click(
        ask,
        inputs=question,
        outputs=answer
    )

demo.launch(debug=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9d84be5b7ec5c7c037.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
